# FlexOn — Contention Generator

This notebook is **only responsible for the external resource environment** used during FlexOn experiments.

It does not invoke `flexon_online`, parse benchmark output, calculate latency statistics, or create experiment plots.

The experiment notebook can therefore be run against either an idle machine or a machine with a contention session active here.

## How the two notebooks work together

1. Configure and start contention in this notebook.
2. Keep this notebook's kernel alive while the contention is active.
3. Run `experiments.ipynb` in another notebook/kernel.
4. Return here and execute the stop cell when the benchmark is complete.

A no-contention experiment simply leaves this session stopped.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "experiments").is_dir():
    PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.contention import ContentionSession, ContentionSpec

print(f"Project root: {PROJECT_ROOT}")

## Configure contention

### CPU contention
`CPU_WORKERS` is the number of busy worker processes. As a practical interpretation, each worker attempts to saturate roughly one logical CPU core.

### GPU contention
- GPU load is intentionally command-based.  

- Set `GPU_COMMAND` to a long-running CUDA workload available on this machine.  

- `GPU_WORKERS` controls how many independent copies of that workload are launched.

The GPU command is left configurable because the experiment harness should not depend on PyTorch, CuPy, or another specific GPU framework.

In [ ]:
# Choose one environment: "none", "cpu", "gpu", or "both".
CONTENTION_KIND = "cpu"

# CPU intensity: number of busy CPU worker processes.
CPU_WORKERS = 4

# GPU intensity: number of copies of the long-running GPU workload.
# Example placeholder — replace with your actual workload command.
GPU_COMMAND = (
    # "python", "path/to/your/gpu_contention_workload.py"
)
GPU_WORKERS = 1

STARTUP_DELAY_SECONDS = 1.0

if CONTENTION_KIND == "none":
    SPEC = ContentionSpec.none(label="none")
elif CONTENTION_KIND == "cpu":
    SPEC = ContentionSpec.cpu(CPU_WORKERS, label=f"cpu-{CPU_WORKERS}workers")
elif CONTENTION_KIND == "gpu":
    if not GPU_COMMAND:
        raise ValueError("Set GPU_COMMAND before selecting gpu contention")
    SPEC = ContentionSpec.gpu(
        GPU_COMMAND,
        workers=GPU_WORKERS,
        label=f"gpu-{GPU_WORKERS}workers",
    )
elif CONTENTION_KIND == "both":
    if not GPU_COMMAND:
        raise ValueError("Set GPU_COMMAND before selecting both contention")
    SPEC = ContentionSpec.both(
        cpu_workers=CPU_WORKERS,
        gpu_command=GPU_COMMAND,
        gpu_workers=GPU_WORKERS,
        label=f"cpu-{CPU_WORKERS}workers+gpu-{GPU_WORKERS}workers",
    )
else:
    raise ValueError(f"Unknown CONTENTION_KIND: {CONTENTION_KIND}")

# Rebuild with the requested startup delay while keeping the selected kind.
SPEC = ContentionSpec(
    cpu_workers=SPEC.cpu_workers,
    gpu_command=SPEC.gpu_command,
    gpu_workers=SPEC.gpu_workers,
    startup_delay_seconds=STARTUP_DELAY_SECONDS,
    label=SPEC.label,
)

print(SPEC)
print("Contention kind:", SPEC.kind)
print("Enabled:", SPEC.enabled)

## Start contention

Execute this cell once. **Do not stop the kernel** while running the experiment notebook.

In [ ]:
session = ContentionSession(SPEC)
session.start()

print("Contention started")
print("kind:", SPEC.kind)
print("CPU workers:", len(session.cpu_processes))
print("GPU workers:", len(session.gpu_processes))

## Check that the session is alive

In [ ]:
print("session active:", session._started)
print("CPU processes alive:", [p.is_alive() for p in session.cpu_processes])
print("GPU processes alive:", [p.poll() is None for p in session.gpu_processes])

## Stop contention

Run this **after all experiments are complete**. The cleanup is also safe to call if the session has already stopped.

In [ ]:
session.stop()
print("Contention stopped")